## I finally got the Docker file on EmperorPhenology jupytr notebook to run 

https://github.com/whoi-mars/EmperorPenguinPhenology

I used the Linux WSL and the docker app to load the exact system that was used when the model was developed. When I tried this previously it did not work so I moved forward with the simple approach that Alex had suggested of using his calculated covariates within the density calculation.


This means that I was able to bypass the depreciated packages and run the model exactly as it was written by Alex. I will then compare those results to the results I got from the modified model that I ran.   

At this stage I believe I will be moving forward with these new results, depending on what they are, for the publication.   

I want to check that the model accurately treats temperatures etc, being that higher density would occur during lower temperatures and higher wind. If that is not the case, there is a larger issue that we need to rectify before we move forward with publication.   

But importantly that issue is engrained in the model development, not a mistake I have made. To test this I will change the temperature and wind speed data to see if that increases or decreases density. 









In [ ]:
#To get the code to run on our data, I input these chunks of code beneath the 
# "Predict count based on meteorological data and colony area" 
# section of his notebook. This uses his TracePhase and the fitted bayesian model using Table S19 (which is data from Sep-Dec and Pt Geologie and Atka Bay)

In [ ]:
df = pd.read_excel("/work/mydata/SAR_datav2_ERA5_v7_cleaned.xlsx")

m = ~df[["area_m2", "temp_airK", "met_rad", "met_ff10", "met_humc"]].isna().any(1)
with phaseTransitionModel(
                 theano.shared(df.loc[m,"temp_airK"].T.values),
                 theano.shared(df.loc[m,"met_ff10"].T.values),
                 theano.shared(df.loc[m,"met_rad"].T.values),
                 theano.shared(df.loc[m,"met_humc"].T.values),
) as model:
    t_area = theano.shared(df.loc[m,"area_m2"].T.values)
    obs = [
        pm.Deterministic("count_pred", t_area*T.maximum(1, model.rho_pred)),
        pm.Deterministic("rho_pred_out", model.rho_pred),
    ]
    phase_ppc = pm.fast_sample_posterior_predictive(tracePhase, var_names=[rv.name for rv in obs], keep_size=True)

df.loc[m, "count_pred"] = phase_ppc["count_pred"].mean((0,1))
df.loc[m, "rho_pred"] = phase_ppc["rho_pred_out"].mean((0,1))

# standard deviation across posterior samples
df.loc[m, "count_pred_sd"] = phase_ppc["count_pred"].std((0,1))
df.loc[m, "rho_pred_sd"] = phase_ppc["rho_pred_out"].std((0,1))

# 95% credible intervals (lower and upper)
import numpy as np
df.loc[m, "count_pred_lower"] = np.percentile(phase_ppc["count_pred"], 2.5, axis=(0,1))
df.loc[m, "count_pred_upper"] = np.percentile(phase_ppc["count_pred"], 97.5, axis=(0,1))
df.loc[m, "rho_pred_lower"] = np.percentile(phase_ppc["rho_pred_out"], 2.5, axis=(0,1))
df.loc[m, "rho_pred_upper"] = np.percentile(phase_ppc["rho_pred_out"], 97.5, axis=(0,1))

In [ ]:
df[["colony_name", "date", "image_id", "phenology_est", "area_m2", "temp_airK", "met_rad", "met_ff10", "met_humc", "count_pred", "count_pred_sd", 
    "count_pred_lower", "count_pred_upper", "rho_pred", "rho_pred_lower", "rho_pred_upper"]].head(20)

In [ ]:
df.to_excel("/work/mydata/SAR_datav2_ERA5_v7_cleaned_with_predictions.xlsx", index=False)

In [ ]:
dfMLR = pd.read_excel("/work/mydata/sar_huddles20230502_ERA5_grouped_v4.xlsx")

m = ~dfMLR[["area_m2", "temp_airK", "met_rad", "met_ff10", "met_humc"]].isna().any(1)
with phaseTransitionModel(
                 theano.shared(dfMLR.loc[m,"temp_airK"].T.values),
                 theano.shared(dfMLR.loc[m,"met_ff10"].T.values),
                 theano.shared(dfMLR.loc[m,"met_rad"].T.values),
                 theano.shared(dfMLR.loc[m,"met_humc"].T.values),
) as model:
    t_area = theano.shared(dfMLR.loc[m,"area_m2"].T.values)
    obs = [
        pm.Deterministic("count_pred", t_area*T.maximum(1, model.rho_pred)),
        pm.Deterministic("rho_pred_out", model.rho_pred),
    ]
    phase_ppc = pm.fast_sample_posterior_predictive(tracePhase, var_names=[rv.name for rv in obs], keep_size=True)

dfMLR.loc[m, "count_pred"] = phase_ppc["count_pred"].mean((0,1))
dfMLR.loc[m, "rho_pred"] = phase_ppc["rho_pred_out"].mean((0,1))

# standard deviation across posterior samples
dfMLR.loc[m, "count_pred_sd"] = phase_ppc["count_pred"].std((0,1))
dfMLR.loc[m, "rho_pred_sd"] = phase_ppc["rho_pred_out"].std((0,1))

# 95% credible intervals (lower and upper)
import numpy as np
dfMLR.loc[m, "count_pred_lower"] = np.percentile(phase_ppc["count_pred"], 2.5, axis=(0,1))
dfMLR.loc[m, "count_pred_upper"] = np.percentile(phase_ppc["count_pred"], 97.5, axis=(0,1))
dfMLR.loc[m, "rho_pred_lower"] = np.percentile(phase_ppc["rho_pred_out"], 2.5, axis=(0,1))
dfMLR.loc[m, "rho_pred_upper"] = np.percentile(phase_ppc["rho_pred_out"], 97.5, axis=(0,1))

In [ ]:
dfMLR[["colony", "date", "image_id",  "area_m2", "temp_airK", "met_rad", "met_ff10", "met_humc", "count_pred", "count_pred_sd", 
    "count_pred_lower", "count_pred_upper", "rho_pred", "rho_pred_lower", "rho_pred_upper"]].head(20)

In [ ]:
df.to_excel("/work/mydata/sar_huddles20230502_ERA5_grouped_v4_withpredictions.xlsx", index=False)